# geoje: Diffusion-TS imputation

Sequence length is fixed to 128. Run cells from top to bottom. Training weights are shared across tasks.


In [ ]:
from pathlib import Path
import json, subprocess, sys, threading
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while ROOT.name != "Diff-ts" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert ROOT.name == "Diff-ts", "Open this notebook from inside the Diff-ts repository."
NAME = "geoje"
TRUTH_NAME = "geoje"
CONFIG = ROOT / "Config/geoje/geoje_128.yaml"
SEQ_LEN = 128
GPU = 0
MILESTONE = 10
TASK = "imputation"
TRAIN_PROPORTION = 0.9
SEEDS = [0, 42, 123]
GPU_BY_SEED = {0: 1, 42: 2, 123: 3}
RUN_TRAINING = False  # Uncond starts training when the training cell runs.
RUN_SAMPLING = False  # Set True after the requested checkpoint exists.
print("repository:", ROOT)
print("config:", CONFIG)

def run_parallel_with_live_output(commands_by_seed, phase):
    """Run seed processes concurrently and stream prefixed output to Jupyter."""
    processes = {}
    threads = []

    def stream(seed, gpu, process):
        prefix = f"[{phase} | seed {seed} | GPU {gpu}]"
        for line in iter(process.stdout.readline, ""):
            print(prefix, line.rstrip(), flush=True)
        process.stdout.close()

    for seed, command in commands_by_seed.items():
        gpu = GPU_BY_SEED[seed]
        print(f"[{phase} | seed {seed} | GPU {gpu}] START", flush=True)
        process = subprocess.Popen(
            command,
            cwd=ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        processes[seed] = process
        thread = threading.Thread(target=stream, args=(seed, gpu, process), daemon=True)
        thread.start(); threads.append(thread)

    return_codes = {seed: process.wait() for seed, process in processes.items()}
    for thread in threads: thread.join()
    for seed, code in return_codes.items():
        status = "DONE" if code == 0 else f"FAILED (exit={code})"
        print(f"[{phase} | seed {seed} | GPU {GPU_BY_SEED[seed]}] {status}", flush=True)
    if any(code != 0 for code in return_codes.values()):
        raise RuntimeError(f"{phase} failed: {return_codes}")
    return return_codes


## 1. Train the shared unconditional diffusion model

This is the README `Training` command. Set `RUN_TRAINING=True` above to execute it.


In [ ]:
def training_command(seed=None):
    gpu = GPU_BY_SEED[seed] if seed is not None else GPU
    cmd = [sys.executable, "-u", "main.py", "--name", NAME, "--config_file", str(CONFIG), "--gpu", str(gpu), "--task", TASK, "--train"]
    if seed is not None:
        cmd += ["--seed", str(seed), "--run_id", f"seed_{seed}"]
    cmd += ["dataloader.train_dataset.params.proportion", str(TRAIN_PROPORTION)]
    return cmd

training_seeds = SEEDS if TASK == "uncond" else [None]
train_commands = [training_command(seed) for seed in training_seeds]
for train_cmd in train_commands: print(" ".join(train_cmd))
if RUN_TRAINING:
    if TASK == "uncond":
        run_parallel_with_live_output(dict(zip(SEEDS, train_commands)), "TRAIN")
        for seed in SEEDS:
            checkpoint_dir = ROOT / "artifacts" / "uncond" / NAME / f"seed_{seed}" / "checkpoints_128"
            final_checkpoint = checkpoint_dir / f"checkpoint-{MILESTONE}.pt"
            assert final_checkpoint.exists(), f"Final checkpoint missing: {final_checkpoint}"
            for milestone in range(1, MILESTONE):
                old_checkpoint = checkpoint_dir / f"checkpoint-{milestone}.pt"
                if old_checkpoint.exists(): old_checkpoint.unlink()
            print(f"seed {seed}: retained checkpoint-{MILESTONE}.pt and removed checkpoints 1-{MILESTONE - 1}")
    else:
        subprocess.run(train_commands[0], cwd=ROOT, check=True)


## 2. Imputation sampling at 75% missingness

This is the README `Imputation` command with the fixed optimized ratio.


In [ ]:
MISSING_RATIO = 0.75
sample_cmd = [sys.executable, "main.py", "--name", NAME, "--config_file", str(CONFIG), "--gpu", str(GPU), "--sample", "1", "--milestone", str(MILESTONE), "--mode", "infill", "--missing_ratio", str(MISSING_RATIO)]
print(" ".join(sample_cmd))
if RUN_SAMPLING:
    subprocess.run(sample_cmd, cwd=ROOT, check=True)


## 3. Evaluate missing positions and visualize reconstruction


In [ ]:
artifact = ROOT / "artifacts" / "imputation" / NAME
prediction_path = artifact / f"ddpm_infill_{NAME}_{SEQ_LEN}.npy"
truth_file = f"sine_ground_truth_{SEQ_LEN}_test.npy" if TRUTH_NAME == "sine" else f"{TRUTH_NAME}_norm_truth_{SEQ_LEN}_test.npy"
truth_path = artifact / "samples" / truth_file
mask_path = artifact / "samples" / f"{TRUTH_NAME}_masking_{SEQ_LEN}.npy"
for path in (prediction_path, truth_path, mask_path): assert path.exists(), f"Missing {path}"
prediction, truth, observed = np.load(prediction_path), np.load(truth_path), np.load(mask_path).astype(bool)
n = min(len(prediction), len(truth), len(observed)); prediction, truth, observed = prediction[:n], truth[:n], observed[:n]
missing = ~observed
error = prediction[missing] - truth[missing]
print({"missing_ratio": float(missing.mean()), "missing_MAE": float(np.mean(np.abs(error))), "missing_RMSE": float(np.sqrt(np.mean(error**2)))})
i, feature = 0, 0
reconstruction = prediction[i, :, feature].copy(); reconstruction[observed[i, :, feature]] = truth[i, observed[i, :, feature], feature]
plt.figure(figsize=(13, 4)); plt.plot(truth[i, :, feature], label="truth")
plt.scatter(np.flatnonzero(observed[i, :, feature]), truth[i, observed[i, :, feature], feature], s=12, label="observed")
plt.plot(reconstruction, label="imputed", alpha=.85); plt.legend(); plt.tight_layout()
